In [570]:
load('IrreducibilityLeanProofWriter.sage')
load('RingOfIntegersLeanProofWriter.sage')
%run CertificateClassGroupLean2.0.ipynb
%run PrimalityCertificate.ipynb
%run InvariantProofs.ipynb


In [269]:
R.<X>= PolynomialRing(ZZ)

#Input defining polynomial. 
T = X^3 - 91
K.<a> = NumberField(T)

#Input basis for the ring of integers. 
B = [1, a, 1/3*a^2 + 1/3*a + 1/3]
B = [K(b) for b in B]
num = '3_1_24843_1'


In [480]:
R.<X>= PolynomialRing(ZZ)

#Input defining polynomial. 
T = X^4 - 2*X^3 + 7*X^2 - 6*X + 78
K.<a> = NumberField(T)

#Input basis for the ring of integers. 
B = [1, a, a^2, 1/35*a^3 + 16/35*a^2 + 3/7*a - 16/35]
B = [K(b) for b in B]
num = '4_0_76176_2'

In [481]:
#Create initial files: 

#Proof of irreducibility of defining polynomial. 
doc = open(f"Irreducible{num}.lean", "w")

#Definition of the number field, definition of subalgebra and basis, and p-maximality proofs. 
f = open(f"RI{num}.lean", "w")

#Write RI (Ring of integers) file. Output is a list of 'bad' primes that do not satisfy Dedekind Criterion, 
# flagl (1 if p unramified), flagW (1 if a wtiness was found), flagD (1 if discriminant ought to be computed) 
bad, flagl, flaglW, flagD = LeanProof(T,B,f,f'Irreducible{num}',num, '',1)

#Write Irreducible file, with proof of irreducibility
LeanProofIrreducible(T, doc)

In [482]:
import contextlib

#List of primes involved in computation of ring of integers. 
F = [factor(T.discriminant())[i][0] for i in range(len(factor(T.discriminant())))] 

#We compute the Minkowski bound: 
M = K.minkowski_bound()

#The number of rational primes below M 
PP = [p for p in primes(floor(M) + 1)]
m = len(PP)

#The number of primes ideals above a single prime is (on average, and assuming Galois group S_n) the harmonic number Hn.  
#Hence, in each lean file with prime ideals we put around 20 ideals so that we consider batches of 20/Hn rational primes. 
NI = ceil(20 / harmonic_number(K.degree()))

#Number of files in which we divide the generation of prime ideals: 
NF = ceil(m/NI)

#Number of intervals we use for the collection certificate. 
number_interval = ceil(NF/2)

t = open(f"PrimesBelowCert{num}.lean", "w")

with contextlib.redirect_stdout(t):
    print(f"""import IdealArithmetic.Examples.NF{num}.PrimesBelow{num}F{NF-1}
noncomputable section""")
    listI, listP = PrimesBelowBoundCertificteGenInterval(K,B,ceil(M),number_interval)
    t.close()

for i in range(NF): 
    f = open(f"PrimesBelow{num}F{i}.lean", "w")
    with contextlib.redirect_stdout(f):
        if i == 0: 
            print(f"""
import IdealArithmetic.IdealArithmetic
import IdealArithmetic.Examples.NF{num}.RI{num}
import IdealArithmetic.ClassGroupGeneration

open Classical Polynomial

noncomputable section """)
            PrimesBelowBound(K, B, PP[i*NI], PP[min((i+1) * NI - 1,len(PP) -1)] , F)
        else : 
            print(f"""
import IdealArithmetic.Examples.NF{num}.PrimesBelow{num}F{i-1}

open Classical Polynomial

noncomputable section """)
            PrimesBelowBound(K, B, PP[i*NI], PP[min((i+1) * NI - 1,len(PP) -1)] , F)
            print("")
    f.close()

In [483]:
#We compute a generator for the class group and the corresponding data to certify its non-principality. 

import re
O = K.ring_of_integers()
Cl = K.class_group('pari')
n = K.degree()

num_gens = len(list(Cl.gens()))

#The generators for each ideal generating the class group. 
ideal_gens = [Cl.gens()[i].ideal().gens_reduced() for i in range(num_gens)]

#The ideals generating the class group. 
J = [O.ideal(ideal_gens[i]) for i in range(num_gens)]

#The order of each of these ideals in the class group. 
t = [Cl(J[i]).order() for i in range(len(Cl.gens()))]

x = [(J[i] ^ t[i]).gens_reduced()[0] for i in range(num_gens)]




In [618]:
pr = 3

In [619]:
phi, A , Q , elems , flag = MatrixPrimesG (K, O, pr, x, t, 200) #flag = 1 for the case that pr divides the torsion order. 
elems = [K(r) for r in elems]

#The primes ideals of degree 1 to certify the non-principality. 
Ql = [list(Q[i].gens()) for i in range(len(Q))]

#The rational prime below these prime ideals. 
Qlq = [Ql[i][0] for i in range(len(Ql))]


if flag == 0 : 
    names = ['zeta' + str(i + 1) for i in range(len(elems) - len(phi))] + [f'alpha{phi[i]}' for i in range(len(phi))]
else : 
    names = ['zeta' + str(i + 1) for i in range(len(elems) - len(phi) - 1)] + ['v'] + [f'alpha{phi[i]}' for i in range(len(phi))]


print(names, elems,x)

['zeta1', 'alpha0'] [-2/35*a^3 + 3/35*a^2 + 1/7*a - 73/35, 2/35*a^3 - 3/35*a^2 + 6/7*a - 67/35] [2/35*a^3 - 3/35*a^2 + 6/7*a - 67/35, 2/35*a^3 - 3/35*a^2 + 6/7*a - 102/35]


In [489]:
#Write the file with the certificate of saturation for the prime p

set_random_seed(10)
l = open(f"ClassGroupSaturated{num}_{pr}.lean", "w")
with contextlib.redirect_stdout(l):
    L, gi = SaturatedCertLean(K, B, num, Ql, pr, elems, names, len(J), ideal_gens, 'J', x, t, flag)
l.close()
print(gi)

[[[1], (2, 1/35*a^3 + 16/35*a^2 - 4/7*a + 89/35), (2, 2/35*a^3 - 3/35*a^2 + 6/7*a + 3/35), (4, -1/35*a^3 + 19/35*a^2 - 3/7*a + 121/35), (4, 2/35*a^3 - 3/35*a^2 + 6/7*a - 67/35), (8, -1/7*a^3 + 5/7*a^2 - 1/7*a + 37/7), (2/35*a^3 - 3/35*a^2 + 6/7*a - 67/35,)], [[1], (6, -1/7*a^3 - 2/7*a^2 - 8/7*a - 12/7), (2/35*a^3 - 3/35*a^2 + 6/7*a - 102/35,)]]


In [535]:
gi

[[[1],
  (2, 1/35*a^3 + 16/35*a^2 - 4/7*a + 89/35),
  (2, 2/35*a^3 - 3/35*a^2 + 6/7*a + 3/35),
  (4, -1/35*a^3 + 19/35*a^2 - 3/7*a + 121/35),
  (4, 2/35*a^3 - 3/35*a^2 + 6/7*a - 67/35),
  (8, -1/7*a^3 + 5/7*a^2 - 1/7*a + 37/7),
  (2/35*a^3 - 3/35*a^2 + 6/7*a - 67/35,)],
 [[1],
  (6, -1/7*a^3 - 2/7*a^2 - 8/7*a - 12/7),
  (2/35*a^3 - 3/35*a^2 + 6/7*a - 102/35,)]]

In [621]:
#CertificateMaximalUnitsNDvdT(A,pr, names[:-len(phi)], 'I', Qlq, 'RC')
#CertificateMaximalUnitsDvdT(A, pr, names[:-len(phi)-1], 'v', 2, 'Sat2','I', Qlq, 'RC')
CertificateMaximalUnitsNDvdT(A,pr, names[:-len(phi)], 'Sat3','I', Qlq, 'RC')

def NPCU3 : pMaximailUnitsCertificateNDvdT O 3 where 
 hp := by decide
 r := 1
 huc := by 
  rw [units_finrank_of_RankUnitsCertificate RC]
  decide
 u := ![zeta1]
 hu := fun i => 
  match i with 
  | 0 => isUnit_zeta1
 t := 2
 hrle := by decide
 q := ![13, 13]
 hqP := by 
  intro i 
  match i with 
  | 0 => exact Sat3.hq13
  | 1 => exact Sat3.hq13
 I := ![Sat3.I0, Sat3.I1]
 hcard := fun i =>
  match i with  
  | 0 => Sat3.N0
  | 1 => Sat3.N1
 ζ := ![2, 2]
 hr := fun i =>
  match i with 
  | 0 => ((orderOf_of_IsOrderOf Sat3.R13) ▸ IsPrimitiveRoot.orderOf _)
  | 1 => ((orderOf_of_IsOrderOf Sat3.R13) ▸ IsPrimitiveRoot.orderOf _)
 hdvd := by decide
 hpndvdt := ne_dvd_torsion3
 M := ![![2], ![2]]
 hM1 := by 
  intro i j
  fin_cases i <;> fin_cases j 
  · erw [eq_of_DiscreteLogCertificate Sat3.Log00] ; decide
  · erw [eq_of_DiscreteLogCertificate Sat3.Log10] ; decide
 Minv := ![![2]]
 hInv := by decide


In [524]:
names

['zeta1', 'alpha0']

In [620]:
#CertificateSaturatedNDvdT(A, pr, names[len(names) - len(phi):], 'J', t, phi, ['J' +str(j)+'_pow'+ str(t[j]) for j in range(len(t))],'NPCU3')
#CertificateSaturatedDvdT(A, pr, names[len(names) - len(phi):], 'J', t, phi, ['J' +str(j)+'_pow'+ str(t[j]) for j in range(len(t))],'NPCU2', 'Sat2')

CertificateSaturatedNDvdT(A, pr, [f'alpha{i}' for i in range(len(t))], 'J', t, phi, ['J' +str(j)+'_pow'+ str(t[j]) for j in range(len(t))],'NPCU3', 'Sat3')

def NPSU3 : pSaturatedClassGroupCertificateNDvdT 3 ![J0, J1] ![6, 2] where 
 topMaximailUnitsCertificateNDvdT := NPCU3
 a := ![alpha0,alpha1]
 γ := 1
 hc := by decide
 ψ := ![0]
 iψ := ![0, 0] 
 hψ := by decide 
 hψn := by 
  intro i 
  fin_cases i 
  · decide
  · convert true_imp_iff.2 ?_ 
    · decide
    · dsimp [alpha1]
      refine (LinearEquiv.map_ne_zero_iff B.equivFun.symm).mpr (by decide) 
 h := fun i =>
  match i with 
  | 0 => J0_pow6
  | 1 => J1_pow2
 N := ![![2], ![1]]
 hM2 := by 
  intro (i : Fin 2) j
  match i , j with 
  | 0, 0 => exact Sat3.Log01
  | 1, 0 => exact Sat3.Log11
 hDl := by decide 
 NInv := ![![1, 1], ![1, 2]]
 hN := by decide 


In [543]:
def CertNames(l): 
    L = ['' for i in range(len(l))] 
    for i in range(len(l)): 
        if l[i] == 0: 
            L[i] = 'dummy'
        elif l[i] == 1 :
            L[i] = 'rfl'
        else: 
            L[i] = 'MulJ' + str(i) + str(l[i] - 2)
    return L 



MAux = [[0 for j in range(t[1])] for i in range(t[0])]

for i in range(t[0]): 
    for j in range(t[1]): 
        if i + j != 0: 
            set_random_seed(10)
            MAux[i][j] = IteratedMulLeanWithCert (K, B, ideal_gens, [i,j], CertNames([i,j]), 'J', gi)


#ExList(str([list(elems_to_basis(g, B).columns()[i]) for i in range(len(g))]))


lemma PowJ0_0J1_1 : J0 ^ 0*J1 ^ 1 = Ideal.span (Set.range fun i ↦ B.equivFun.symm (![![6, 0, 0, 0], ![-4, 1, 2, -5]] i)) := by 
 simp only [pow_succ, pow_one, pow_zero, one_mul, mul_one]
 rfl

lemma PowJ0_1J1_0 : J0 ^ 1*J1 ^ 0 = Ideal.span (Set.range fun i ↦ B.equivFun.symm (![![2, 0, 0, 0], ![3, -1, 0, 1]] i)) := by 
 simp only [pow_succ, pow_one, pow_zero, one_mul, mul_one]
 rfl
def MulRJ0_1J1_1 : IdealMulEqCertificate O ℤ timesTableO (J0) J1
  ![![2, 0, 0, 0], ![3, -1, 0, 1]] ![![6, 0, 0, 0], ![-4, 1, 2, -5]]
  ![![6, 0, 0, 0], ![0, -1, -1, 3]] where
 T := Table
 heq := timesTableT_eq_Table
 hI1 := rfl
 hI2 := rfl
 M :=  ![![![12, 0, 0, 0], ![-8, 2, 4, -10]], ![![18, -6, 0, 6], ![2, 13, 11, -23]]]
 hmul := by decide
 f :=  ![![![![11597040, 6949866, 9332922, -19547995], ![12827931, -9055080, -7140, 0]], ![![1179, 22, 0, 0], ![0, 0, 0, 0]]], ![![![-70090440, -42003766, -56406536, 118144641], ![-77529712, 54727298, 43154, 0]], ![![-7125, -132, 0, 0], ![1, 0, 0, 0]]]]
 g :=  ![![![![0

In [544]:
#Names for the proofs of J_1^l_1...J_m^l_m  

def name_cert(l) : 
    if l == [0 for i in range(len(l))]:
        return 'rfl'
    else: 
        return 'exact Pow' + ''.join(['J' + str(i) + '_' + str(l[i]) for i in range(len(l))])

In [545]:
def generators_vec(l): 
    return MAux[l[0]][l[1]]
    

In [586]:
primes_below_gens_list = [0 for i in range(100)]
for i in range(100): 
    if is_prime(i): 
        primes_below_gens_list[i] = PrimesBelowGens(K,i)

In [587]:
def primes_below_gens(p): 
    return primes_below_gens_list[p]

In [608]:
primes_below_gens(3)[1][0][1] = 1/35*a^3 + 16/35*a^2 - 11/7*a + 54/35 

In [603]:
2 * B[0] -2 * B[1] + 1 * B[3]

#[2, -2, 0, 1]

1/35*a^3 + 16/35*a^2 - 11/7*a + 54/35

In [609]:
# We generate the file with the relations between the prime ideals of norm below M and the generator of the 
# class group J. 
import os
os.environ['OMP_NUM_THREADS'] = '1'

set_random_seed(10)
r = open(f"RelationIdeals{num}.lean", "w")
with contextlib.redirect_stdout(r):
    print(f"""import IdealArithmetic.Examples.NF{num}.PrimesBelow{num}F{NF - 1}
import IdealArithmetic.Examples.NF{num}.ClassGroupSaturated{num}_{pr}

noncomputable section""")
    BM = RelationsGenerator(K, B, ceil(M), ideal_gens, 'J', name_cert, generators_vec,primes_below_gens)
r.close()

In [611]:
%%capture capInv

print(f"""import IdealArithmetic.Examples.NF{num}.ClassGroupSaturated{num}_{pr}
import IdealArithmetic.Examples.NF{num}.RelationIdeals{num}
import IdealArithmetic.Examples.NF{num}.PrimesBelowCert{num}
import Mathlib.NumberTheory.NumberField.ClassNumber
import IdealArithmetic.ResultantRecursive
import IdealArithmetic.DiscriminantSubalgebraBuilder
import IdealArithmetic.ClassGroupGeneration

/- Number field `K(α)` with with α root of polynomial `{T}`.
Class number `{t}`, generated by class of ideal `J = ({ideal_gens[0]}, {ideal_gens[1]})`. We have `J ^ {pr}` is principal. -/

/- Ring of integers with basis `{B}` -/

open BigOperators Classical Matrix Polynomial

noncomputable section

instance hirr : Fact $ (Irreducible (map (algebraMap ℤ ℚ) T)) where
  out :=  (Polynomial.Monic.irreducible_iff_irreducible_map_fraction_map (T_monic)).1 T_irreducible """)

print(f"""
noncomputable instance K_field : Field K := by
  unfold K
  infer_instance

instance K_numberField : NumberField K := by
  unfold K
  infer_instance

lemma K_finrank : Module.finrank ℚ K = {len(B)} := by
  unfold K
  rw [Module.finrank_eq_card_basis (AdjoinRoot.powerBasisAux _), Polynomial.natDegree_map_eq_of_injective, T_degree]
  · simp
  · exact RingHom.injective_int (algebraMap ℤ ℚ)
  · exact Irreducible.ne_zero hirr.out """)

print(f"""
theorem O_integral_closure : O = integralClosure ℤ K := by
  refine eq_of_piMaximal_at_all_primes_int O Om hm ?_
  intro p hp
  by_cases hc : p ∈ {bad}
  · fin_cases hc""")
for p in bad:
    if flagl[p] == 0:
        if flaglW[p] == 1:
            print(f'    exact pMaximal_of_MaximalOrderCertificateWLists {p} O Om hm M{p}')
        else : 
            print(f'    exact pMaximal_of_MaximalOrderCertificateLists {p} O Om hm M{p}')
    else : 
        print(f'    exact pMaximal_of_MaximalOrderCertificateOfUnramifiedLists {p} O Om hm M{p}')
print(f"""  · haveI : Fact $ Nat.Prime p := fact_iff.2 hp
    refine piMaximal_of_root_in_order_of_satisfiesDedekindCriterion_int Adj T_monic hm ?_ hroot_mem
     (satisfiesDedekindAlmostAllLists_of_certificate T _ T_ofList {bad} D p hp hc)
    rw [T_degree, rank_subalgebra_eq_card_basis Om B']
""")
print("")
print(f"""theorem  O_ringOfIntegers' : O = NumberField.RingOfIntegers K := by rw [O_integral_closure] ; rfl
""")

print("""

instance : Module.Finite ℤ (Additive ((↥O)ˣ ⧸ CommGroup.torsion (↥O)ˣ)) := by
  rw [O_integral_closure]
  exact NumberField.Units.instFiniteIntAdditiveQuotientUnitsRingOfIntegersSubgroupTorsion K

instance : Module.Free ℤ (Additive ((↥O)ˣ ⧸ CommGroup.torsion (↥O)ˣ)) := by
  rw [O_integral_closure]
  exact NumberField.Units.instFreeIntAdditiveQuotientUnitsRingOfIntegersSubgroupTorsion K

instance :  Fintype ↥(CommGroup.torsion (↥O)ˣ) := by
  rw [O_integral_closure]
  exact NumberField.Units.instFintypeSubtypeUnitsRingOfIntegersMemSubgroupTorsion K

instance : IsCyclic ↥(CommGroup.torsion (↥O)ˣ) := by
  rw [O_integral_closure]
  exact NumberField.Units.instIsCyclicSubtypeUnitsRingOfIntegersMemSubgroupTorsion K

instance DD' : IsDedekindDomain O  := by
  rw [O_integral_closure]
  exact integralClosure.isDedekindDomain ℤ ℚ K

instance : Module.Free ℤ ↥O := Module.Free.of_basis B 

""")
RankUnitCertificateLean(R(T), 'T_ofList', 'Adj', 'O_integral_closure', 'RC')

if flagD == 1:
    D = K.discriminant()
    print(f"""
lemma T_discr : T.discriminant = {T.discriminant()} :=  by
  convert discriminant_eq_DiscriminantOfPRemainder_of_SturmBuilderOfList SturmRC
  rw [T_ofList]

theorem K_discr : NumberField.discr K = {D} := by
  rw [discr_numberField_eq_discrSubalgebraBuilder T_irreducible BQ O_integral_closure]
  rw [T_discr]
  rfl

lemma K_nrComplexPlaces : NumberField.InfinitePlace.nrComplexPlaces K = {K.signature()[1]} := by
  rw [nrComplexPlaces_of_RankUnitsCertificate RC]
  rfl

theorem K_minowski : MinkowskiBound K ≤ ({float(K.minkowski_bound() + 0.01)} : ℝ) := by
  refine K_minkowski_decimal _ ?_
  rw [K_nrComplexPlaces, K_discr, K_finrank]
  have hraux: √{D} ≤ {round(sqrt(D),6) + 0.00001} := by
      refine Real.sqrt_le_iff.mpr ?_
      norm_num
  have : {factorial(K.degree())/(K.degree() ^ K.degree())} * √{round(sqrt(D),6) + 0.00001} ≤ {round((factorial(K.degree())/(K.degree() ^ K.degree())) * (round(sqrt(D),6) + 0.00001), 6)} := by
    refine le_trans (mul_le_mul_of_nonneg (by rfl) hraux (by norm_num) (by norm_num)) ?_
    norm_num
  norm_num
  linarith

""")


In [612]:
with open(f"Invariants{num}.lean", "w") as f:
    f.write(capInv.stdout)
f.close()

In [623]:
#Construct the vector of ideals and the corresponding proofs of the generation of the class group and saturation. 

#I is the list of (names) of ideals (with multiplicity) below the Minkowski bound. F is the list flagging if the corresponding ideal is principal (1) or 
# not. BM[i][j] is the exponent of the i-th ideal corresponding to the j-th generator of the class group. 
# name_primes_cert corresponds to the name of the certificate for the primes below bound B. 
# B is the bound 
# J_name: name for the generator of the class group. Default is J. 
# group_str is the list describing the structure of the class group. The i-th entry is the order of J_i ideal in the class group. 
# ideal_pow_proof is the name of the proof of J^q = (a) 
# names_sat_cert: list of pairs [n,f] where f is the name of the saturation certificates for the corresponding p (in increasing order), and f is
# flag, 0 if p does not divide the torsion order and 1 if it does. 

def ClassGroupOrderProof(I,F, BM, name_primes_cert,B, J_names,group_str, names_sat_cert):
    Idedup = []
    Fdedup = []
    for i in range(len(I)):
        if I[i] not in Idedup: 
            Idedup.append(I[i])
            Fdedup.append(F[i])
    n = len(Idedup)
    I_dedup_string = "[" + ", ".join(Idedup) + "]" 
    I_string = "[" + ", ".join(I) + "]" 
    F = factor(prod(group_str))
    prime_list = [F[i][0] for i in range(len(F))]
    exp_list = [F[i][1] for i in range(len(F))]
    print(f'''def g : Fin {n} → Ideal O := !{I_dedup_string}''')
    print('''''')
    print(f'''lemma primesGenerationBelowAux {{x}} : x ∈ (List.ofFn {name_primes_cert}.Il).flatten ↔ x ∈ List.ofFn g := by
  have h1: (List.ofFn {name_primes_cert}.Il).flatten = {I_string} := by rfl
  have h2: List.ofFn g = {I_dedup_string} := by rfl
  rw [h1, h2]
  simp only [List.mem_cons, List.not_mem_nil, or_false, or_self, or_self_left] ''')
    print('')
    print(f'''lemma primesGenerationBelow : {{I : Ideal O | 0 < I.absNorm ∧ I.IsPrime ∧ I.absNorm < {B}}} ⊆ Set.range g := by
  refine le_primes_below_bound_of_PrimesBelowBoundCertificate' (Subalgebra.equivOfEq _ _ O_integral_closure).toRingEquiv g {name_primes_cert} ?_
  intro x hx
  exact primesGenerationBelowAux.1 hx ''')
    print('')
    print(f"def BM : Matrix (Fin {n}) (Fin {len(BM[0])}) ℕ := {ExList(str(BM))} ")
    print('')
    print(f'''def g' : Fin {n} → nonZeroDivisors (Ideal O) := by
  intro i
  have : g i ∈ (List.ofFn {name_primes_cert}.Il).flatten := by 
    simp only [primesGenerationBelowAux, List.mem_ofFn, exists_apply_eq_apply]
  exact Ideal.toNonZeroDivisorOfNeZero (g i) 
    (fun hc => ((zero_ne_mem_of_PrimesBelowCertificate _ {name_primes_cert}) (hc ▸ this) ))
    
def x : Fin {len(BM[0])} → Ideal O := ![{','.join(J_names)}] 
    
def x' :  Fin {len(BM[0])} → nonZeroDivisors (Ideal O) := by 
  refine fun i => Ideal.toNonZeroDivisorOfNeZero (x i) (?_ )
  sorry 

lemma g'_apply : ∀ (i : Fin {n}), ↑(g' i) = g i := by
  intro i
  rfl

lemma x'_apply : ∀ (i : Fin {len(BM[0])}), ↑(x' i) = x i := by
  intro i
  rfl

theorem class_group_generator : Subgroup.closure (Set.range (fun i => ClassGroup.mk0 (x' i))) = ⊤ := by
    refine subgroup_closure_eq_classGroup'' (Subalgebra.equivOfEq _ _ O_integral_closure).toRingEquiv
      g'_apply x'_apply ?_ primesGenerationBelow BM ?_
    · refine lt_of_le_of_lt K_minowski ?_
      norm_num
    · intro i
      simp only [Fin.isValue, Fin.prod_univ_succ, Finset.univ_eq_empty, Finset.prod_empty, Fin.succ_zero_eq_one, mul_one]
      fin_cases i ''')
    for i in range(len(Idedup)): 
        if Fdedup[i] != 1:
            print(f'''      · refine Exists.intro _ (Exists.intro ?_ (Exists.intro (?_) (Exists.intro ?_ (by convert R{(Idedup[i])[1:]}))))
        refine Nat.cast_ne_zero.2 (by decide)
        exact (LinearEquiv.map_ne_zero_iff B.equivFun.symm).mpr (by decide)''')
        else: 
            print('''      · exact ideal_mem_principal_class' O ℤ B _ _ (by decide) rfl''')
    print('')
    print(f'''def ClassGroupO_equiv : (∀ i : Fin 2 , (ZMod (!{group_str} i))) ≃+ Additive (ClassGroup O) := by
  haveI : ∀ (i : Fin {len(group_str)}), NeZero (!{group_str} i) := by
    intro i
    fin_cases i <;>  exact {{out := by decide}}
  refine ClassGroup_equiv_of_pSaturatedCertificate x'_apply {names_sat_cert[0][0]}.h class_group_generator ?_
  rintro ⟨p, hp1, hp2⟩
  have : ∏ i, !{group_str} i = ∏ i, (!{prime_list} i) ^ (!{exp_list} i)  := by decide
  rw [this] at hp2
  have hp_cases : p ∈ {prime_list} := by
    obtain ⟨j ,hj1, hj2⟩  := Prime.exists_mem_finset_dvd (Nat.prime_iff.mp hp1) hp2
    show p ∈ List.ofFn !{prime_list}
    rw [(Nat.prime_dvd_prime_iff_eq hp1 ?_).1 (Nat.Prime.dvd_of_dvd_pow hp1 hj2), List.mem_ofFn]
    exact ⟨j, rfl⟩ 
    fin_cases j <;> decide
  simp only [List.mem_cons, List.not_mem_nil, or_false] at hp_cases ''')
    for i in range(len(prime_list)): 
        print(f'''  by_cases h : p = {prime_list[i]}
  simp_rw [h] ; {['right' if names_sat_cert[i][1] == 0 else 'left' for i in range(len(names_sat_cert))][i]}
  exact {names_sat_cert[i][0]} ''')
    print(f'''  omega 

def class_group_equiv : (∀ i : Fin {len(group_str)} , (ZMod (!{group_str} i))) ≃+ Additive (ClassGroup (NumberField.RingOfIntegers K)) := by
 refine AddEquiv.trans ClassGroupO_equiv ?_
 exact MulEquiv.toAdditive (ClassGroup.congr
    ((Subalgebra.equivOfEq _ _ O_integral_closure).toRingEquiv))

theorem class_number_K_eq_{prod(group_str)} : NumberField.classNumber K = {prod(group_str)} := by
  unfold NumberField.classNumber
  rw [Fintype.card_eq_nat_card, ← Nat.card_congr (Additive.toMul),
    Nat.card_congr (class_group_equiv).symm.toEquiv, Nat.card_pi]
  simp only [Fin.isValue, Fin.prod_univ_two]
  simp''')
    

In [626]:
#ClassGroupPrimeOrderProof(listI, listP, BM, f'PB{ceil(M)}', ceil(M), ['J0', 'J1'])
ClassGroupOrderProof(listI, listP, BM, f'PB{ceil(M)}', ceil(M), ['J0', 'J1'], [6,2], [['NPSU2', 1], ['NPSU3', 0]])

def g : Fin 12 → Ideal O := ![I2N0, I2N1, I3N0, I3N1, I5N0, I5N1, I13N0, I13N1, I13N2, I13N3, I23N0, I23N1]

lemma primesGenerationBelowAux {x} : x ∈ (List.ofFn PB42.Il).flatten ↔ x ∈ List.ofFn g := by
  have h1: (List.ofFn PB42.Il).flatten = [I2N0, I2N0, I2N1, I2N1, I3N0, I3N0, I3N1, I3N1, I5N0, I5N1, I13N0, I13N1, I13N2, I13N3, I23N0, I23N0, I23N1, I23N1] := by rfl
  have h2: List.ofFn g = [I2N0, I2N1, I3N0, I3N1, I5N0, I5N1, I13N0, I13N1, I13N2, I13N3, I23N0, I23N1] := by rfl
  rw [h1, h2]
  simp only [List.mem_cons, List.not_mem_nil, or_false, or_self, or_self_left] 

lemma primesGenerationBelow : {I : Ideal O | 0 < I.absNorm ∧ I.IsPrime ∧ I.absNorm < 42} ⊆ Set.range g := by
  refine le_primes_below_bound_of_PrimesBelowBoundCertificate' (Subalgebra.equivOfEq _ _ O_integral_closure).toRingEquiv g PB42 ?_
  intro x hx
  exact primesGenerationBelowAux.1 hx 

def BM : Matrix (Fin 12) (Fin 2) ℕ := ![![5, 0], ![1, 0], ![4, 1], ![2, 1], ![3, 1], ![3, 1], ![5, 1], ![1, 1], ![5, 1], ![1, 1]

In [428]:
print(f''''theorem class_group_generator : Subgroup.closure (Set.range (fun i => ClassGroup.mk0 (x' i))) = ⊤ := by
    refine subgroup_closure_eq_classGroup'' (r := Fin 1) (Subalgebra.equivOfEq _ _ O_integral_closure).toRingEquiv
      g'_apply x'_apply ?_ primesGenerationBelow BM ?_
    · refine lt_of_le_of_lt K_minowski ?_
      norm_num
    · simp only [Finset.univ_unique, Fin.default_eq_zero, Fin.isValue, Finset.prod_singleton]
      intro i
      fin_cases i ''')
    for i in range(len(Idedup)): 
        if Fdedup[i] != 1:
            print(f'''      · refine Exists.intro _ (Exists.intro ?_ (Exists.intro (?_) (Exists.intro ?_ (by convert R{(Idedup[i])[1:]}))))
        refine Nat.cast_ne_zero.2 (by decide)
        exact (LinearEquiv.map_ne_zero_iff B.equivFun.symm).mpr (by decide)''')
        else: 
            print('''      · exact ideal_mem_principal_class' O ℤ B _ _ (by decide) rfl''')
    print('')

IndentationError: unexpected indent (3381099370.py, line 9)

In [613]:
pNeDvdTorsionLean(K, B, 'K_finrank' , 'O_integral_closure', 'IC', pr)

def IC : Ideal O := Ideal.span (Set.range (fun i ↦ B.equivFun.symm (![![2, 0, 0, 0], ![0, 1, 0, 0]] i)))

def AIC: IdealEqSpanCertificate O ℤ timesTableO IC
 ![![2, 0, 0, 0], ![0, 1, 0, 0]] 
 ![![2, 0, 0, 0], ![0, 1, 0, 0], ![0, 0, 1, 0], ![0, 0, 0, 1]] where
  T := Table 
  heq := timesTableT_eq_Table
  hieq := rfl 
  M :=![![![2, 0, 0, 0], ![0, 2, 0, 0], ![0, 0, 2, 0], ![0, 0, 0, 2]], ![![0, 1, 0, 0], ![0, 0, 1, 0], ![16, -15, -16, 35], ![6, -8, -8, 18]]]
  hmulB := by decide
  f := ![![![1, 0, 0, 0], ![0, 0, 0, 0]], ![![0, 0, 0, 0], ![1, 0, 0, 0]], ![![0, 0, 0, 0], ![0, 1, 0, 0]], ![![8, -12, -7, 18], ![9, -2, -1, 0]]]
  g := ![![![1, 0, 0, 0], ![0, 2, 0, 0], ![0, 0, 2, 0], ![0, 0, 0, 2]], ![![0, 1, 0, 0], ![0, 0, 1, 0], ![8, -15, -16, 35], ![3, -8, -8, 18]]]
  hle1 := by decide
  hle2 := by decide

 lemma ne_dvd_torsion3 : ¬3 ∣ Nat.card ↥(CommGroup.torsion (↥O)ˣ) := by 
  refine prime_not_dvd_torsion_of_not_dvd (by decide) IC 
     (ideal_norm_eq_prod' B _ _ (by decide) 0 0 (by dec

In [439]:
['right' if [1,0,0,1][i] == 0 else 'left' for i in range(len([1,0,0,1]))]

['left', 'right', 'right', 'left']

In [468]:
round(sqrt(23),3)

4.796

In [469]:
factorial(K.degree())/(K.degree() ^ K.degree())

2/9

In [470]:
round(factorial(K.degree())/(K.degree() ^ K.degree()) * round(sqrt(23),6), 6)

1.06574

In [471]:
round(1.45,1)

1.4